In [39]:
import numpy as np
import pandas as pd
import yfinance as yf
import sys
sys.path.append("../notebooks")
from utils import *# update_list_of_tickers, create_updated_list, read_updated_list
filtre_tickers = pd.read_csv('../src/data-pipline/filterd_strong_cap_financials.csv')

tickers=filtre_tickers['Ticker'].tolist()

In [40]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AFL',
 'A',
 'ALLE',
 'ALL',
 'GOOGL',
 'GOOG',
 'AMZN',
 'AME',
 'AMGN',
 'APH',
 'APA',
 'ACGL',
 'AIZ',
 'ADP',
 'AVY',
 'BKR',
 'BBY',
 'BLK',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BF-B',
 'BLDR',
 'CHRW',
 'CDNS',
 'CAT',
 'CBOE',
 'CDW',
 'CF',
 'CB',
 'CHD',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'CLX',
 'CME',
 'KO',
 'CTSH',
 'COIN',
 'CL',
 'CMCSA',
 'COP',
 'STZ',
 'CEG',
 'GLW',
 'CPAY',
 'COST',
 'CSX',
 'CMI',
 'DECK',
 'DAL',
 'DVN',
 'DXCM',
 'DOV',
 'EMN',
 'ETN',
 'EBAY',
 'ECL',
 'EW',
 'EA',
 'ELV',
 'ENPH',
 'EOG',
 'EFX',
 'ERIE',
 'EXPE',
 'FDS',
 'FAST',
 'FSLR',
 'FTNT',
 'FOXA',
 'FOX',
 'FCX',
 'IT',
 'GE',
 'GEHC',
 'GD',
 'GILD',
 'GL',
 'GDDY',
 'HAL',
 'HIG',
 'HSY',
 'HD',
 'HON',
 'HWM',
 'HUBB',
 'HII',
 'IBM',
 'IEX',
 'IDXX',
 'ITW',
 'INCY',
 'PODD',
 'IPG',
 'JBHT',
 'JBL',
 'JKHY',
 'JNJ',
 'K',
 'KVUE',
 'KMB',
 'KLAC',
 'KR',
 'LRCX',
 'LVS',
 'LDOS',
 'LII',
 'LLY',
 'LIN',
 'LYV',
 'LMT',
 'MKTX',
 '

In [2]:
df=pd.read_csv("..\\src\\data-pipline\\filterd_strong_cap_financials.csv")




In [42]:
len(tickers)

200

In [43]:


HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
def to_float(value):
    """Convert to float safely."""
    try:
        if value is None:
            return None
        if isinstance(value, str):
            value = value.replace(",", "").strip()  # remove commas
            if value.upper() in ["N/A", "NONE", ""]:
                return None
        return float(value)
    except:
        return None

def to_datetime(value):
    """Convert to datetime safely."""
    try:
        if value is None:
            return None
        if isinstance(value, (int, float)):
            return pd.to_datetime(value, unit='s')
        if isinstance(value, str):
            return pd.to_datetime(value, errors='coerce')
        return None
    except:
        return None
# ---------------- SEC functions ----------------
def get_cik(ticker):
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?CIK={ticker}&owner=exclude&action=getcompany&output=atom"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        return None
    soup = BeautifulSoup(resp.content, "xml")
    cik_tag = soup.find("cik")
    return cik_tag.text.strip() if cik_tag else None

def get_latest_10k_url(cik):
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=10-K&owner=exclude&count=10&output=atom"
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        return None
    soup = BeautifulSoup(resp.content, "xml")
    entry = soup.find("entry")
    if entry:
        link = entry.find("link")["href"]
        txt_url = link.replace("-index.htm", ".txt")
        return txt_url
    return None

def get_fiscal_year_from_text(filing_url):
    resp = requests.get(filing_url, headers=HEADERS)
    if resp.status_code != 200:
        return None
    text = resp.text
    patterns = [
        r"fiscal year (?:ended|ends) (\w+ \d{1,2}, \d{4})",
        r"for the year ended (\w+ \d{1,2}, \d{4})",
        r"year ended (\w+ \d{1,2}, \d{4})",
        r"fiscal year ended (\w+ \d{1,2}, \d{4})",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    return None

# ---------------- YFinance function ----------------
def get_yf_fundamentals(ticker):
    """Fetch company fundamentals from yfinance with type-safe conversion."""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        return {
            "Company_Name": info.get("longName"),
            "Sector": info.get("sector"),
            "Industry": info.get("industry"),
            "Market_Cap": to_float(info.get("marketCap")),
            "Beta": to_float(info.get("beta")),
            "Relative_to_market_volatility": to_float(info.get("beta")),
            "Dividend_Yield": to_float(info.get("dividendYield")),
            "Ex_dividend_date": to_datetime(info.get("exDividendDate")),
            "Dividend_pay_date": to_datetime(info.get("lastDividendDate")),
        }
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return {
            "Company_Name": None,
            "Sector": None,
            "Industry": None,
            "Market_Cap": None,
            "Beta": None,
            "Relative_to_market_volatility": None,
            "Dividend_Yield": None,
            "Ex_dividend_date": None,
            "Dividend_pay_date": None,
        }
# ---------------- Main function ----------------
def fetch_company_data(tickers):
    records = []
    for ticker in tickers:
        print(f"Processing {ticker}...")

        # SEC Fiscal Year
        fiscal_year = None
        try:
            cik = get_cik(ticker)
            if cik:
                filing_url = get_latest_10k_url(cik)
                fiscal_year = get_fiscal_year_from_text(filing_url) if filing_url else None
        except:
            pass

        # YFinance fundamentals
        yf_data = get_yf_fundamentals(ticker)

        # Combine data
        records.append({
            "Ticker": ticker,
            "Fiscal_Year_End": fiscal_year,
            **yf_data
        })

        # Pause to avoid SEC blocking
        time.sleep(1)

    return pd.DataFrame(records)

# ---------------- Example usage ----------------
#tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META"]
df = fetch_company_data(tickers)
print(df)


Processing MMM...
Processing AOS...
Processing ABT...
Processing ABBV...
Processing ACN...
Processing ADBE...
Processing AFL...
Processing A...
Processing ALLE...
Processing ALL...
Processing GOOGL...
Processing GOOG...
Processing AMZN...
Processing AME...
Processing AMGN...
Processing APH...
Processing APA...
Processing ACGL...
Processing AIZ...
Processing ADP...
Processing AVY...
Processing BKR...
Processing BBY...
Processing BLK...
Processing BSX...
Processing BMY...
Processing AVGO...
Processing BR...
Processing BF-B...
Processing BLDR...
Processing CHRW...
Processing CDNS...
Processing CAT...
Processing CBOE...
Processing CDW...
Processing CF...
Processing CB...
Processing CHD...
Processing CI...
Processing CINF...
Processing CTAS...
Processing CSCO...
Processing CLX...
Processing CME...
Processing KO...
Processing CTSH...
Processing COIN...
Processing CL...
Processing CMCSA...
Processing COP...
Processing STZ...
Processing CEG...
Processing GLW...
Processing CPAY...
Processing CO

In [44]:
df.to_excel("../src/data-pipline/company_fundamentals.xlsx", index=False)

In [23]:
exl=pd.read_excel("../src/data-pipline/company_fundamentals.xlsx")


In [46]:
tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AFL',
 'A',
 'ALLE',
 'ALL',
 'GOOGL',
 'GOOG',
 'AMZN',
 'AME',
 'AMGN',
 'APH',
 'APA',
 'ACGL',
 'AIZ',
 'ADP',
 'AVY',
 'BKR',
 'BBY',
 'BLK',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BF-B',
 'BLDR',
 'CHRW',
 'CDNS',
 'CAT',
 'CBOE',
 'CDW',
 'CF',
 'CB',
 'CHD',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'CLX',
 'CME',
 'KO',
 'CTSH',
 'COIN',
 'CL',
 'CMCSA',
 'COP',
 'STZ',
 'CEG',
 'GLW',
 'CPAY',
 'COST',
 'CSX',
 'CMI',
 'DECK',
 'DAL',
 'DVN',
 'DXCM',
 'DOV',
 'EMN',
 'ETN',
 'EBAY',
 'ECL',
 'EW',
 'EA',
 'ELV',
 'ENPH',
 'EOG',
 'EFX',
 'ERIE',
 'EXPE',
 'FDS',
 'FAST',
 'FSLR',
 'FTNT',
 'FOXA',
 'FOX',
 'FCX',
 'IT',
 'GE',
 'GEHC',
 'GD',
 'GILD',
 'GL',
 'GDDY',
 'HAL',
 'HIG',
 'HSY',
 'HD',
 'HON',
 'HWM',
 'HUBB',
 'HII',
 'IBM',
 'IEX',
 'IDXX',
 'ITW',
 'INCY',
 'PODD',
 'IPG',
 'JBHT',
 'JBL',
 'JKHY',
 'JNJ',
 'K',
 'KVUE',
 'KMB',
 'KLAC',
 'KR',
 'LRCX',
 'LVS',
 'LDOS',
 'LII',
 'LLY',
 'LIN',
 'LYV',
 'LMT',
 'MKTX',
 '

In [72]:
import yfinance as yf
import pandas as pd
import numpy as np

def get_eps_metrics(ticker):
    try:
        tk = yf.Ticker(ticker)

        # ---- Get Shares Outstanding ----
        shares = None

        # First try fast_info (much more reliable)
        try:
            shares = tk.fast_info.shares_outstanding
        except:
            pass

        # Fall back to .info
        if shares is None:
            shares = tk.info.get("sharesOutstanding", None)

        if shares is None:
            return {"Ticker": ticker, "Error": "Shares outstanding not available"}

        # ---- Get Income Statement ----
        fin = tk.financials
        if fin is None or fin.empty:
            return {"Ticker": ticker, "Error": "Financial data missing"}

        net_income = fin.loc["Net Income"]

        # ---- Get EPS Values ----
        eps_annual = (net_income / shares).dropna()

        if eps_annual.empty:
            return {"Ticker": ticker, "Error": "Annual EPS data missing"}

        # Most recent 3 years
        eps_annual = eps_annual.sort_index(ascending=False)

        # Build output safely
        output = {
            "Ticker": ticker,
            "EPS Annual": eps_annual.to_dict(),
        }

        return output

    except Exception as e:
        return {"Ticker": ticker, "Error": str(e)}


In [73]:
for t in ["AAPL", "MSFT", "META", "INTC", "MMM"]:
    print(get_eps_metrics(t))


{'Ticker': 'AAPL', 'EPS Annual': {Timestamp('2025-09-30 00:00:00'): 7.5803549089548685, Timestamp('2024-09-30 00:00:00'): 6.343649207622477, Timestamp('2023-09-30 00:00:00'): 6.5642043067054505, Timestamp('2022-09-30 00:00:00'): 6.7542376660871595}}
{'Ticker': 'MSFT', 'EPS Annual': {Timestamp('2025-06-30 00:00:00'): 13.70113370537547, Timestamp('2024-06-30 00:00:00'): 11.858385578766718, Timestamp('2023-06-30 00:00:00'): 9.735915390591115, Timestamp('2022-06-30 00:00:00'): 9.786639400793474}}
{'Ticker': 'META', 'EPS Annual': {Timestamp('2024-12-31 00:00:00'): 28.633228000904392, Timestamp('2023-12-31 00:00:00'): 17.95224420108018, Timestamp('2022-12-31 00:00:00'): 10.652515869483354, Timestamp('2021-12-31 00:00:00'): 18.077135766446535}}
{'Ticker': 'INTC', 'EPS Annual': {Timestamp('2024-12-31 00:00:00'): -3.9320754716981132, Timestamp('2023-12-31 00:00:00'): 0.3540880503144654, Timestamp('2022-12-31 00:00:00'): 1.680083857442348, Timestamp('2021-12-31 00:00:00'): 4.165199161425576}}
{'

In [77]:
import yfinance as yf
import pandas as pd
import numpy as np

def get_eps_last5(ticker_symbol):
    try:
        ticker = yf.Ticker(ticker_symbol)

        # --- Get shares outstanding ---
        shares = ticker.info.get("sharesOutstanding")
        if shares is None:
            return {"Ticker": ticker_symbol, "Error": "Missing shares outstanding"}

        # --- Get annual financials ---
        fin = ticker.financials
        if fin is None or fin.empty:
            return {"Ticker": ticker_symbol, "Error": "Missing annual financials"}

        # --- Compute EPS ---
        eps = fin.loc["Net Income"] / shares

        # --- Convert index to year only ---
        eps.index = eps.index.year.astype(int)

        # --- Keep last 5 years (most recent) ---
        last5_years = sorted(eps.index)[-5:]

        # --- Build flat dictionary ---
        row = {"Ticker": ticker_symbol}
        for year in last5_years:
            row[f"EPS_{year}"] = float(eps[year]) if year in eps.index else np.nan

        return row

    except Exception as e:
        return {"Ticker": ticker_symbol, "Error": str(e)}


In [78]:
for t in ["AAPL", "MSFT", "META", "INTC", "MMM"]:
    print(get_eps_last5(t))

{'Ticker': 'AAPL', 'EPS_2021': nan, 'EPS_2022': 6.7542376660871595, 'EPS_2023': 6.5642043067054505, 'EPS_2024': 6.343649207622477, 'EPS_2025': 7.5803549089548685}
{'Ticker': 'MSFT', 'EPS_2022': 9.786639400793474, 'EPS_2023': 9.735915390591115, 'EPS_2024': 11.858385578766718, 'EPS_2025': 13.70113370537547}
{'Ticker': 'META', 'EPS_2020': nan, 'EPS_2021': 18.077135766446535, 'EPS_2022': 10.652515869483354, 'EPS_2023': 17.95224420108018, 'EPS_2024': 28.633228000904392}
{'Ticker': 'INTC', 'EPS_2020': nan, 'EPS_2021': 4.165199161425576, 'EPS_2022': 1.680083857442348, 'EPS_2023': 0.3540880503144654, 'EPS_2024': -3.9320754716981132}
{'Ticker': 'MMM', 'EPS_2020': nan, 'EPS_2021': 11.145935272238894, 'EPS_2022': 10.874863716893108, 'EPS_2023': -13.167677289192884, 'EPS_2024': 7.8554277809581}


In [10]:
def get_company_fundamentals(ticker):
    url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={ticker}&apikey={API_KEY}"
    data = requests.get(url).json()

    if "Symbol" not in data:
        return {"error": "No data returned. Check API key, ticker, or API limits.", "raw": data}

    return {
        "ticker": ticker,
        "fiscal_year_end": data.get("FiscalYearEnd"),
        "beta": data.get("Beta"),
        "relative_volatility": data.get("Beta"),
        "dividend_yield": data.get("DividendYield"),
        "ex_dividend_date": data.get("ExDividendDate"),
        "dividend_per_share": data.get("DividendPerShare"),
        "sector": data.get("Sector"),
        "market_cap": data.get("MarketCapitalization")
    }
for ticker in tickers:
    print(get_company_fundamentals(ticker))

{'ticker': 'META', 'fiscal_year_end': 'December', 'beta': '1.272', 'relative_volatility': '1.272', 'dividend_yield': '0.0035', 'ex_dividend_date': '2025-09-22', 'dividend_per_share': '2.075', 'sector': 'COMMUNICATION SERVICES', 'market_cap': '1487917941000'}
{'ticker': 'NFLX', 'fiscal_year_end': 'December', 'beta': '1.703', 'relative_volatility': '1.703', 'dividend_yield': 'None', 'ex_dividend_date': 'None', 'dividend_per_share': 'None', 'sector': 'COMMUNICATION SERVICES', 'market_cap': '466105565000'}
{'ticker': 'GOOGL', 'fiscal_year_end': 'December', 'beta': '1.082', 'relative_volatility': '1.082', 'dividend_yield': '0.0036', 'ex_dividend_date': '2025-12-08', 'dividend_per_share': '1.02', 'sector': 'COMMUNICATION SERVICES', 'market_cap': '3546550370000'}
{'ticker': 'AMZN', 'fiscal_year_end': 'December', 'beta': '1.368', 'relative_volatility': '1.368', 'dividend_yield': 'None', 'ex_dividend_date': 'None', 'dividend_per_share': 'None', 'sector': 'CONSUMER CYCLICAL', 'market_cap': '2380

In [39]:
import requests
import pandas as pd


def get_company_fundamentals(ticker):
    url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={ticker}&apikey={API_KEY}"
    data = requests.get(url).json()

    if "Symbol" not in data:
        return {"error": "No data returned. Check API key, ticker, or API limits.", "raw": data}

    return {
        "ticker": ticker,
        "fiscal_year_end": data.get("FiscalYearEnd"),
        "beta": data.get("Beta"),
        "relative_volatility": data.get("Beta"),
        "dividend_yield": data.get("DividendYield"),
        "ex_dividend_date": data.get("ExDividendDate"),
        "dividend_per_share": data.get("DividendPerShare"),
        "sector": data.get("Sector"),
        "market_cap": data.get("MarketCapitalization")
    }

# Example: single ticker
ticker_data = []
for ticker in tickers:
    ticker_data.append(get_company_fundamentals(ticker))    



In [40]:
ticker_data


[{'ticker': 'META',
  'fiscal_year_end': 'December',
  'beta': '1.272',
  'relative_volatility': '1.272',
  'dividend_yield': '0.0035',
  'ex_dividend_date': '2025-09-22',
  'dividend_per_share': '2.075',
  'sector': 'COMMUNICATION SERVICES',
  'market_cap': '1487917941000'},
 {'ticker': 'NFLX',
  'fiscal_year_end': 'December',
  'beta': '1.703',
  'relative_volatility': '1.703',
  'dividend_yield': 'None',
  'ex_dividend_date': 'None',
  'dividend_per_share': 'None',
  'sector': 'COMMUNICATION SERVICES',
  'market_cap': '466105565000'},
 {'ticker': 'GOOGL',
  'fiscal_year_end': 'December',
  'beta': '1.082',
  'relative_volatility': '1.082',
  'dividend_yield': '0.0036',
  'ex_dividend_date': '2025-12-08',
  'dividend_per_share': '1.02',
  'sector': 'COMMUNICATION SERVICES',
  'market_cap': '3546550370000'},
 {'ticker': 'AMZN',
  'fiscal_year_end': 'December',
  'beta': '1.368',
  'relative_volatility': '1.368',
  'dividend_yield': 'None',
  'ex_dividend_date': 'None',
  'dividend_per

In [ ]:
import pandas as pd
import yfinance as yf
import requests

def get_fmp_fiscal_end(ticker, api_key="AUlO5yj7Y9sscxWN8UW5wDFE1FK2a3Tt"):
    url = f"https://financialmodelingprep.com/api/v4/company-core-information?symbol={ticker}&apikey={api_key}"
    r = requests.get(url)
    if r.status_code == 200:
        data = r.json()
        if data and isinstance(data, list):
            return data[0].get("fiscalYearEnd")
    return None

def get_company_data(tickers, fmp_api_key=None):
    rows = []
    for symbol in tickers:
        info = yf.Ticker(symbol).info
        fy_end = info.get("fiscalYearEnd")
        # fallback
        if fy_end is None and fmp_api_key:
            fy_end = get_fmp_fiscal_end(symbol, api_key=fmp_api_key)

        row = {
            "Company": info.get("longName"),
            "Ticker": symbol,
            "Fiscal Year End": fy_end,
            "Beta": info.get("beta"),
            "Relative market volatility": info.get("beta"),
            "Dividend Yield": info.get("dividendYield"),
            "Ex-Dividend Date": pd.to_datetime(info.get("exDividendDate"), unit="s", errors="coerce"),
            "Dividend Pay Date": pd.to_datetime(info.get("dividendDate"), unit="s", errors="coerce"),
        }
        rows.append(row)
    return pd.DataFrame(rows)

# Example usage
tickers = ["AAPL","MSFT","TSLA","AMZN"]
df = get_company_data(tickers, fmp_api_key=FMP_API_KEY)
print(df)


                 Company Ticker Fiscal Year End   Beta  \
0             Apple Inc.   AAPL            None  1.109   
1  Microsoft Corporation   MSFT            None  1.065   
2            Tesla, Inc.   TSLA            None  1.872   
3       Amazon.com, Inc.   AMZN            None  1.368   

   Relative market volatility  Dividend Yield Ex-Dividend Date  \
0                       1.109            0.39       2025-11-10   
1                       1.065            0.75       2025-11-20   
2                       1.872             NaN              NaT   
3                       1.368             NaN              NaT   

  Dividend Pay Date  
0        2025-11-13  
1        2025-12-11  
2               NaT  
3               NaT  


In [30]:
# Example usage
FMP_API_KEY="AUlO5yj7Y9sscxWN8UW5wDFE1FK2a3Tt"
tickers = ["AAPL","MSFT","TSLA","AMZN"]
df = get_company_data(tickers, FMP_API_KEY)
print(df)

TypeError: get_company_data() takes 1 positional argument but 2 were given

In [2]:
tickers=read_updated_list(ticker_csv)


Reading List of S&P 500 Tickers

Number of tickers: 503
First 10: ['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']
✓ Successfully read 503 tickers from data-pipline/sp500_tickers.csv



In [ ]:
import pandas as pd
#df_=pd.read_csv('C:\\Users\\IdrisEl-Feghi\\OneDrive - Batsalani\\Desktop\\market-predictor\\src\\data-pipline\\large_cap_financials.csv')

df_=pd.read_csv("../src/data-pipline/large_cap_financials.csv")


In [12]:
summarize_filters(df_)

,Metric,Count,% of Total
0,Total Companies,502,100.0%
1,Passed All Filters,200,39.8%
2,Failed Operating Margin,12,2.4%
3,Failed ROE,212,42.2%
4,Failed Leverage (Net Debt/EBITDA),178,35.5%
5,Failed Interest Coverage,103,20.5%


In [13]:
filtered_df =  filter_strong_tickers(df_)
len(filtered_df)
#filtered_df.to_csv("../src/data-pipline/filterd_strong_cap_financials.csv")

[INFO] 200 tickers passed the filter criteria


200

In [10]:
df_.head()

,Ticker,Revenue,Operating_Margin,Net_Margin,ROE,Total_Debt,Total_Cash,EBITDA,Interest_Expense
0,MMM,24824999936,0.24367,0.13700,0.72921,1.317900e+10,5.188000e+09,6.161000e+09,1.191000e+09
1,AOS,3830099968,0.18631,0.13851,0.28209,2.225000e+08,1.728000e+08,7.843000e+08,6.700000e+06
2,ABT,43842998272,0.19395,0.31880,0.30620,1.297500e+10,7.733000e+09,1.174700e+10,5.590000e+08
3,ABBV,59643998208,0.35497,0.04004,1.37961,6.884900e+10,5.671000e+09,2.951900e+10,2.808000e+09
4,ACN,69672976384,0.15220,0.11021,0.25509,8.182866e+09,1.148467e+10,1.222253e+10,2.285550e+08
